# Strobemer Types Comparison

This notebook compares different strobemer types (randstrobes, minstrobes, hybridstrobes) and allows you to adjust all parameters for testing.

In [ ]:
import sys
from pathlib import Path
from collections import defaultdict
import pandas as pd
import matplotlib.pyplot as plt

# Add parent directory to path for imports
sys.path.insert(0, str(Path.cwd().parent))

from strobemers.modules import indexing

## Helper Functions

In [ ]:
def generate_strobes(seq, k_size=15, w_min=20, w_max=70, w=1, order=2, strobe_type='randstrobes'):
    """
    Generate strobemers from a DNA sequence.
    
    Parameters
    ----------
    seq : str
        DNA sequence
    k_size : int
        Total strobemer length in nucleotides
    w_min : int
        Minimum window offset
    w_max : int
        Maximum window offset
    w : int
        Thinning window (w=1: all, w>1: sparse)
    order : int
        Strobemer order (2 or 3)
    strobe_type : str
        Type of strobemer: 'randstrobes', 'minstrobes', or 'hybridstrobes'
    
    Returns
    -------
    dict
        Dictionary with position tuples as keys and hash values as values
    """
    if strobe_type == 'randstrobes':
        return indexing.randstrobes(seq, k_size, w_min, w_max, w, order=order)
    elif strobe_type == 'minstrobes':
        return indexing.minstrobes(seq, k_size, w_min, w_max, w, order=order)
    elif strobe_type == 'hybridstrobes':
        return indexing.hybridstrobes(seq, k_size, w_min, w_max, w, order=order)
    else:
        raise ValueError(f"Unknown strobe_type: {strobe_type}")

In [ ]:
def parse_fasta(fasta_file):
    """Parse FASTA file and yield (seq_id, sequence) tuples."""
    seq_id = None
    seq = ""
    
    with open(fasta_file, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if seq_id and seq:
                    yield seq_id, seq
                seq_id = line[1:].split()[0]
                seq = ""
            else:
                seq += line.upper()
        
        if seq_id and seq:
            yield seq_id, seq

In [ ]:
def compare_strobe_types(seq, seq_id="test", k_size=15, w_min=20, w_max=70, w=1, order=2):
    """
    Compare all three strobemer types on the same sequence.
    
    Returns
    -------
    dict
        Results for each strobemer type
    """
    results = {}
    
    for strobe_type in ['randstrobes', 'minstrobes', 'hybridstrobes']:
        results[strobe_type] = generate_strobes(
            seq, k_size, w_min, w_max, w, order, strobe_type
        )
    
    return results

In [ ]:
def print_comparison(results, seq_id="test"):
    """Print comparison table for different strobemer types."""
    print("=" * 70)
    print(f"STROBEMER TYPE COMPARISON: {seq_id}")
    print("=" * 70)
    print(f"{'Type':<20} {'Count':>10} {'Unique Hashes':>15}")
    print("-" * 70)
    
    for strobe_type, strobes in results.items():
        unique_hashes = len(set(strobes.values()))
        print(f"{strobe_type:<20} {len(strobes):>10} {unique_hashes:>15}")
    
    print("=" * 70)

In [ ]:
def extract_strobe_sequences(seq, positions, k_size):
    """Extract actual strobemer sequences from positions."""
    if len(positions) == 2:
        p1, p2 = positions
        s1 = seq[p1:p1 + k_size//2]
        s2 = seq[p2:p2 + k_size//2]
        return f"{s1}+{s2}"
    elif len(positions) == 3:
        p1, p2, p3 = positions
        s1 = seq[p1:p1 + k_size//3]
        s2 = seq[p2:p2 + k_size//3]
        s3 = seq[p3:p3 + k_size//3]
        return f"{s1}+{s2}+{s3}"
    return str(positions)

## Test 1: Compare Strobemer Types on a Single Sequence

In [ ]:
# Create a test sequence
test_seq = "ACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGT" * 10
print(f"Test sequence length: {len(test_seq)} bp")

# Compare all types with default parameters
results = compare_strobe_types(
    test_seq, 
    seq_id="test_sequence",
    k_size=15, 
    w_min=20, 
    w_max=70, 
    w=1, 
    order=2
)

print_comparison(results, "test_sequence")

In [ ]:
# Show sample strobemers from each type
print("\nSAMPLE STROBEMERS (first 5 from each type):\n")

for strobe_type, strobes in results.items():
    print(f"{strobe_type.upper()}:")
    for i, (positions, hash_val) in enumerate(list(strobes.items())[:5]):
        seq_str = extract_strobe_sequences(test_seq, positions, 15)
        if len(positions) == 2:
            print(f"  #{i+1}: pos={positions}, hash={hash_val:>15}, seq={seq_str}")
        else:
            print(f"  #{i+1}: pos={positions}, hash={hash_val:>15}, seq={seq_str}")
    print()

## Test 2: Parameter Sweep - Vary k_size

In [ ]:
def parameter_sweep(seq, param_name, param_values, fixed_params=None, strobe_type='randstrobes'):
    """
    Sweep through parameter values and collect results.
    
    Parameters
    ----------
    seq : str
        DNA sequence
    param_name : str
        Parameter to sweep: 'k_size', 'w_min', 'w_max', 'w', or 'order'
    param_values : list
        List of parameter values to test
    fixed_params : dict
        Fixed parameters for other values
    strobe_type : str
        Type of strobemer to test
    
    Returns
    -------
    pd.DataFrame
        Results table
    """
    if fixed_params is None:
        fixed_params = {'k_size': 15, 'w_min': 20, 'w_max': 70, 'w': 1, 'order': 2}
    
    results = []
    
    for value in param_values:
        params = fixed_params.copy()
        params[param_name] = value
        
        strobes = generate_strobes(seq, **params, strobe_type=strobe_type)
        unique_hashes = len(set(strobes.values()))
        
        results.append({
            param_name: value,
            'count': len(strobes),
            'unique_hashes': unique_hashes
        })
    
    return pd.DataFrame(results)

In [ ]:
# Test with varying k_size
k_values = [10, 12, 15, 18, 20, 24, 30]
k_sweep = parameter_sweep(
    test_seq, 
    'k_size', 
    k_values,
    fixed_params={'w_min': 20, 'w_max': 70, 'w': 1, 'order': 2},
    strobe_type='randstrobes'
)

print("\nVarying k_size (randstrobes):")
print(k_sweep.to_string(index=False))

# Plot
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(k_sweep['k_size'], k_sweep['count'], 'o-', linewidth=2)
plt.xlabel('k_size')
plt.ylabel('Number of Strobemers')
plt.title('Effect of k_size on Strobemer Count')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(k_sweep['k_size'], k_sweep['unique_hashes'], 's-', color='orange', linewidth=2)
plt.xlabel('k_size')
plt.ylabel('Unique Hashes')
plt.title('Effect of k_size on Uniqueness')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Test 3: Parameter Sweep - Vary Window Parameters

In [ ]:
# Test with varying w_min
w_min_values = [10, 15, 20, 25, 30, 40, 50]
w_min_sweep = parameter_sweep(
    test_seq, 
    'w_min', 
    w_min_values,
    fixed_params={'k_size': 15, 'w_max': 70, 'w': 1, 'order': 2},
    strobe_type='randstrobes'
)

print("\nVarying w_min (randstrobes, k=15, w_max=70):")
print(w_min_sweep.to_string(index=False))

# Plot
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(w_min_sweep['w_min'], w_min_sweep['count'], 'o-', linewidth=2)
plt.xlabel('w_min')
plt.ylabel('Number of Strobemers')
plt.title('Effect of w_min on Strobemer Count')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(w_min_sweep['w_min'], w_min_sweep['unique_hashes'], 's-', color='green', linewidth=2)
plt.xlabel('w_min')
plt.ylabel('Unique Hashes')
plt.title('Effect of w_min on Uniqueness')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Test with varying w_max
w_max_values = [40, 50, 60, 70, 80, 100, 150]
w_max_sweep = parameter_sweep(
    test_seq, 
    'w_max', 
    w_max_values,
    fixed_params={'k_size': 15, 'w_min': 20, 'w': 1, 'order': 2},
    strobe_type='randstrobes'
)

print("\nVarying w_max (randstrobes, k=15, w_min=20):")
print(w_max_sweep.to_string(index=False))

# Plot
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(w_max_sweep['w_max'], w_max_sweep['count'], 'o-', linewidth=2)
plt.xlabel('w_max')
plt.ylabel('Number of Strobemers')
plt.title('Effect of w_max on Strobemer Count')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(w_max_sweep['w_max'], w_max_sweep['unique_hashes'], 's-', color='red', linewidth=2)
plt.xlabel('w_max')
plt.ylabel('Unique Hashes')
plt.title('Effect of w_max on Uniqueness')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Test 4: Parameter Sweep - Vary Thinning (w)

In [ ]:
# Test with varying thinning window
w_values = [1, 2, 5, 10, 20, 50, 100]
w_sweep = parameter_sweep(
    test_seq, 
    'w', 
    w_values,
    fixed_params={'k_size': 15, 'w_min': 20, 'w_max': 70, 'order': 2},
    strobe_type='randstrobes'
)

print("\nVarying thinning window w (randstrobes, k=15, w_min=20, w_max=70):")
print(w_sweep.to_string(index=False))

# Plot
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.semilogx(w_sweep['w'], w_sweep['count'], 'o-', linewidth=2)
plt.xlabel('w (thinning window, log scale)')
plt.ylabel('Number of Strobemers')
plt.title('Effect of Thinning on Strobemer Count')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.semilogx(w_sweep['w'], w_sweep['unique_hashes'], 's-', color='purple', linewidth=2)
plt.xlabel('w (thinning window, log scale)')
plt.ylabel('Unique Hashes')
plt.title('Effect of Thinning on Uniqueness')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Test 5: Compare All Types with Different Orders

In [ ]:
def compare_all_types_and_orders(seq, k_size=15, w_min=20, w_max=70, w=1):
    """Compare all strobemer types with order 2 and 3."""
    results = []
    
    for order in [2, 3]:
        for strobe_type in ['randstrobes', 'minstrobes', 'hybridstrobes']:
            try:
                strobes = generate_strobes(seq, k_size, w_min, w_max, w, order, strobe_type)
                unique_hashes = len(set(strobes.values()))
                results.append({
                    'order': order,
                    'type': strobe_type,
                    'count': len(strobes),
                    'unique_hashes': unique_hashes
                })
            except Exception as e:
                results.append({
                    'order': order,
                    'type': strobe_type,
                    'count': 0,
                    'unique_hashes': 0,
                    'error': str(e)
                })
    
    return pd.DataFrame(results)

In [ ]:
# Compare all types and orders
comparison_df = compare_all_types_and_orders(test_seq, k_size=15, w_min=20, w_max=70, w=1)

print("\nCOMPARISON OF ALL STROBEMER TYPES AND ORDERS:")
print("=" * 70)
print(comparison_df.to_string(index=False))
print("=" * 70)

# Plot
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
for order in [2, 3]:
    subset = comparison_df[comparison_df['order'] == order]
    plt.bar(range(len(subset)), subset['count'], alpha=0.7, label=f'Order {order}')
plt.xticks(range(len(comparison_df[comparison_df['order']==2])), 
           ['rand\n2', 'min\n2', 'hybrid\n2', 'rand\n3', 'min\n3', 'hybrid\n3'])
plt.ylabel('Number of Strobemers')
plt.title('Strobemer Count by Type and Order')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
for order in [2, 3]:
    subset = comparison_df[comparison_df['order'] == order]
    plt.bar(range(len(subset)), subset['unique_hashes'], alpha=0.7, label=f'Order {order}')
plt.xticks(range(len(comparison_df[comparison_df['order']==2])), 
           ['rand\n2', 'min\n2', 'hybrid\n2', 'rand\n3', 'min\n3', 'hybrid\n3'])
plt.ylabel('Unique Hashes')
plt.title('Unique Hashes by Type and Order')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Test 6: Process FASTA File with Custom Parameters

In [ ]:
def process_fasta_with_params(fasta_file, k_size=15, w_min=20, w_max=70, w=1, order=2, strobe_type='randstrobes', max_seqs=10):
    """
    Process FASTA file with custom parameters.
    
    Parameters
    ----------
    fasta_file : str
        Path to FASTA file
    k_size : int
        Total strobemer length
    w_min : int
        Minimum window offset
    w_max : int
        Maximum window offset
    w : int
        Thinning window
    order : int
        Strobemer order
    strobe_type : str
        Type of strobemer
    max_seqs : int
        Maximum number of sequences to process
    
    Returns
    -------
    pd.DataFrame
        Results table
    """
    results = []
    
    for i, (seq_id, seq) in enumerate(parse_fasta(fasta_file)):
        if i >= max_seqs:
            break
        
        strobes = generate_strobes(seq, k_size, w_min, w_max, w, order, strobe_type)
        unique_hashes = len(set(strobes.values()))
        
        results.append({
            'seq_id': seq_id[:50],
            'seq_length': len(seq),
            'strobemer_count': len(strobes),
            'unique_hashes': unique_hashes,
            'strobe_type': strobe_type,
            'k_size': k_size,
            'w_min': w_min,
            'w_max': w_max,
            'w': w,
            'order': order
        })
    
    return pd.DataFrame(results)

In [ ]:
# Process actual SIRV data
data_dir = Path.cwd().parent / "strobemers" / "data"
fasta_file = data_dir / "ONT_sirv_cDNA_seqs.fasta"

if fasta_file.exists():
    print(f"Processing: {fasta_file}")
    
    # Test with different parameters
    df_default = process_fasta_with_params(
        fasta_file, 
        k_size=15, w_min=20, w_max=70, w=1, order=2, 
        strobe_type='randstrobes',
        max_seqs=20
    )
    
    print("\nResults with default parameters (randstrobes, k=15, w_min=20, w_max=70):")
    print(df_default[['seq_id', 'seq_length', 'strobemer_count', 'unique_hashes']].to_string(index=False))
    
    # Summary statistics
    print("\nSummary Statistics:")
    print(f"  Mean strobemers per sequence: {df_default['strobemer_count'].mean():.1f}")
    print(f"  Mean unique hashes: {df_default['unique_hashes'].mean():.1f}")
    print(f"  Mean sequence length: {df_default['seq_length'].mean():.1f}")
    
    # Plot
    plt.figure(figsize=(10, 5))
    plt.scatter(df_default['seq_length'], df_default['strobemer_count'], alpha=0.6)
    plt.xlabel('Sequence Length')
    plt.ylabel('Number of Strobemers')
    plt.title('Strobemer Count vs Sequence Length')
    plt.grid(True, alpha=0.3)
    plt.show()
    
else:
    print(f"FASTA file not found: {fasta_file}")

## Test 7: Interactive Parameter Testing

In [ ]:
def interactive_test(seq, **params):
    """
    Interactive test function - adjust parameters and see results.
    
    Usage:
        interactive_test(test_seq, k_size=20, w_min=25, w_max=80, order=3)
    """
    # Default parameters
    defaults = {
        'k_size': 15,
        'w_min': 20,
        'w_max': 70,
        'w': 1,
        'order': 2,
        'strobe_type': 'randstrobes'
    }
    
    # Update with provided parameters
    defaults.update(params)
    
    print("=" * 70)
    print("STROBEMER GENERATION TEST")
    print("=" * 70)
    print(f"Parameters:")
    for key, value in defaults.items():
        print(f"  {key}: {value}")
    print("=" * 70)
    
    # Generate strobemers
    strobe_type = defaults.pop('strobe_type')
    strobes = generate_strobes(seq, **defaults, strobe_type=strobe_type)
    
    print(f"\nResults:")
    print(f"  Total strobemers: {len(strobes)}")
    print(f"  Unique hashes: {len(set(strobes.values()))}")
    print(f"  Sequence length: {len(seq)} bp")
    
    # Show first 5
    print(f"\nFirst 5 strobemers:")
    for i, (positions, hash_val) in enumerate(list(strobes.items())[:5]):
        seq_str = extract_strobe_sequences(seq, positions, defaults['k_size'])
        print(f"  #{i+1}: pos={positions}, hash={hash_val}, seq={seq_str}")
    
    print("=" * 70)
    
    return strobes

In [ ]:
# Example: Test with custom parameters
# Uncomment and modify to test different parameters

# Test 1: Default parameters
result1 = interactive_test(test_seq)

# Test 2: Order 3 strobemers
result2 = interactive_test(test_seq, order=3, k_size=18)

# Test 3: Hybridstrobes with different window
result3 = interactive_test(test_seq, strobe_type='hybridstrobes', w_min=30, w_max=100)

# Test 4: Minstrobes with thinning
result4 = interactive_test(test_seq, strobe_type='minstrobes', w=10)

## Summary Table

In [ ]:
# Create comprehensive comparison
summary_data = []

test_params = [
    {'k_size': 15, 'w_min': 20, 'w_max': 70, 'w': 1, 'order': 2},
    {'k_size': 20, 'w_min': 25, 'w_max': 80, 'w': 1, 'order': 2},
    {'k_size': 15, 'w_min': 20, 'w_max': 70, 'w': 10, 'order': 2},
    {'k_size': 18, 'w_min': 20, 'w_max': 70, 'w': 1, 'order': 3},
]

for params in test_params:
    for strobe_type in ['randstrobes', 'minstrobes', 'hybridstrobes']:
        strobes = generate_strobes(test_seq, **params, strobe_type=strobe_type)
        summary_data.append({
            'strobe_type': strobe_type,
            'k_size': params['k_size'],
            'w_min': params['w_min'],
            'w_max': params['w_max'],
            'w': params['w'],
            'order': params['order'],
            'count': len(strobes),
            'unique_hashes': len(set(strobes.values()))
        })

summary_df = pd.DataFrame(summary_data)

print("\nCOMPREHENSIVE PARAMETER COMPARISON:")
print("=" * 100)
print(summary_df.to_string(index=False))
print("=" * 100)

## Key Takeaways

1. **Strobemer Types**: Randstrobes, minstrobes, and hybridstrobes produce similar counts but different hash distributions
2. **k_size**: Larger k = fewer strobemers, more unique hashes
3. **w_min/w_max**: Window placement affects mutation tolerance
4. **Thinning (w)**: Higher w = fewer strobemers (sparser sampling)
5. **Order**: Order 3 provides more specificity but uses more memory